In [1]:

import rasterio as rio
from scipy import ndimage
import numpy as np
import cupy as cp
import ctypes

/home/dhester/ms-land-cover/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def mode(arr):
    """Return the mode of a numpy array."""
    return np.bincount(arr.astype(int)).argmax()

In [13]:

data_path = '/home/dhester/server/guser/dh/LC_tests_v4/MS/105/lc_classes.tif'
with rio.open(data_path) as src:
    data = src.read(1)
    profile = src.profile
    cmap = src.colormap(1)
    
# filtered_data = ndimage.generic_filter(data, mode, size=3, mode='reflect')

In [9]:
from scipy.ndimage import convolve

def fast_mode_filter_known_classes(data: np.ndarray, classes: list[int], size: int = 3) -> np.ndarray:
    kernel = np.ones((size, size), dtype=np.uint8)
    counts = np.stack([convolve((data == c).astype(np.uint8), kernel, mode='reflect') for c in classes], axis=0)
    return np.array(classes)[np.argmax(counts, axis=0)]

filtered_data = fast_mode_filter_known_classes(data, classes=[0, 1, 2, 3, 4, 5, 6,7, 8], size=5)

out_data_path = data_path.replace('.tif', '_filtered_5.tif')
with rio.open(out_data_path, 'w', **profile) as dst:
    dst.write(filtered_data, 1)
    dst.write_colormap(1, cmap)

NameError: name 'cmap' is not defined

In [11]:
with rio.open(out_data_path, 'w', **profile) as dst:
    dst.write(filtered_data, 1)
    dst.write_colormap(1, cmap)

In [7]:
from joblib import Parallel, delayed
from scipy.ndimage import convolve

def fast_mode_filter_parallel(data: np.ndarray, classes: list[int], size: int = 3) -> np.ndarray:
    kernel = np.ones((size, size), dtype=np.uint8)
    
    def process_class(c):
        return convolve((data == c).astype(np.uint8), kernel, mode='reflect')
    
    counts = np.stack(Parallel(n_jobs=-1)(delayed(process_class)(c) for c in classes), axis=0)
    return np.array(classes)[np.argmax(counts, axis=0)]

filtered_data = fast_mode_filter_parallel(data, classes=[0, 1, 2, 3, 4, 5, 6,7, 8], size=7)
out_data_path = data_path.replace('.tif', '_filtered_7.tif')
with rio.open(out_data_path, 'w', **profile) as dst:
    dst.write(filtered_data, 1)
    dst.write_colormap(1, cmap)

CPLE_AppDefinedError: Deleting /home/dhester/server/guser/dh/LC_tests_v3/MS/001/lc_classes_filtered_7.tif failed: Device or resource busy

In [ ]:
from joblib import Parallel, delayed

def fast_mode_filter_gpu(data: np.ndarray, classes: list[int], size: int = 3) -> np.ndarray:
    kernel = cp.ones((size, size), dtype=cp.uint8)
    data_gpu = cp.asarray(data)
    counts = cp.stack([cp.convolve((data_gpu == c).astype(cp.uint8), kernel, mode='reflect') 
                     for c in classes], axis=0)
    result = cp.array(classes)[cp.argmax(counts, axis=0)]
    return cp.asnumpy(result)

def fast_mode_filter_gpu_optimized(data: np.ndarray, classes: list[int], size: int = 3) -> np.ndarray:
    # Use memory pool for more efficient memory allocation
    with cp.cuda.Device(0):
        # Create 1D kernel for separable convolution
        kernel_1d = cp.ones(size, dtype=cp.uint8)
        data_gpu = cp.asarray(data)
        
        # Pre-allocate results array
        counts = cp.zeros((len(classes), *data.shape), dtype=cp.int32)
        
        # Process each class with separable convolution
        for i, c in enumerate(classes):
            # Create binary mask
            mask = (data_gpu == c).astype(cp.uint8)
            # Apply horizontal convolution
            temp = cp.convolve(mask, kernel_1d.reshape(1, -1), mode='reflect')
            # Apply vertical convolution and store result
            counts[i] = cp.convolve(temp, kernel_1d.reshape(-1, 1), mode='reflect')
        
        # Get result and transfer back to CPU only once
        result = cp.array(classes)[cp.argmax(counts, axis=0)]
        return cp.asnumpy(result)


def fast_mode_filter_gpu_batched(data: np.ndarray, classes: list[int], size: int = 3, batch_size: int = 3) -> np.ndarray:
    kernel = cp.ones((size, size), dtype=cp.uint8)
    data_gpu = cp.asarray(data)
    result_shape = data.shape
    
    # Process classes in batches to optimize memory usage
    counts = None
    for i in range(0, len(classes), batch_size):
        batch_classes = classes[i:i+batch_size]
        batch_counts = cp.stack([cp.convolve((data_gpu == c).astype(cp.uint8), kernel, mode='reflect') 
                              for c in batch_classes], axis=0)
        
        if counts is None:
            counts = batch_counts
            batch_indices = cp.arange(len(batch_classes))
        else:
            # Compare with previous best counts
            batch_max = cp.argmax(batch_counts, axis=0)
            batch_max_values = cp.take_along_axis(batch_counts, batch_max[cp.newaxis], axis=0)[0]
            
            # Get current best counts
            current_max = cp.argmax(counts, axis=0)
            current_max_values = cp.take_along_axis(counts, current_max[cp.newaxis], axis=0)[0]
            
            # Update where batch is better
            mask = batch_max_values > current_max_values
            current_max[mask] = batch_indices[batch_max[mask]] + i
            
            # Update counts for the next comparison
            counts = cp.maximum(counts, batch_counts)
    
    result = cp.array(classes)[cp.argmax(counts, axis=0)]
    return cp.asnumpy(result)



ValueError: v cannot be multidimensional array

In [ ]:
for kernel_size in range(3, 12, 2):
    
    print(f'Processing kernel size {kernel_size}')
    filtered_data = fast_mode_filter_parallel(data, classes=list(range(1, 9)), size=kernel_size)
    out_data_path = data_path.replace('.tif', f'_filtered_{kernel_size}.tif')
    print(f'Writing to {out_data_path}')
    with rio.open(out_data_path, 'w', **profile) as dst:
        dst.write(filtered_data, 1)
        dst.write_colormap(1, cmap)

Processing kernel size 3
